In [1]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# load processed df
from IPython.utils.capture import capture_output

with capture_output():
     %run ../feature_engineering/feature_selection.ipynb

# Decision Trees

In [ ]:
# === Model 1: Decision Tree (SETUP) ===
import numpy as np
import pandas as pd
import time

from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier

# Uses your backward-selected feature matrix + labels
Xb = X_backward
y_arr = np.asarray(y)

dt_model = DecisionTreeClassifier(
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=42
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
# === Model 1: Decision Tree (EVALUATION) ===
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

dt_fold_times, dt_fold_auc, dt_fold_acc, dt_fold_f1 = [], [], [], []

t0_total = time.perf_counter()

for fold, (train_idx, test_idx) in enumerate(cv.split(Xb, y_arr), start=1):
    X_train, X_test = Xb.iloc[train_idx], Xb.iloc[test_idx]
    y_train, y_test = y_arr[train_idx], y_arr[test_idx]

    t0 = time.perf_counter()
    dt_model.fit(X_train, y_train)
    dt_fold_times.append(time.perf_counter() - t0)

    y_proba = dt_model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)

    dt_fold_auc.append(roc_auc_score(y_test, y_proba))
    dt_fold_acc.append(accuracy_score(y_test, y_pred))
    dt_fold_f1.append(f1_score(y_test, y_pred))

total_train_time = time.perf_counter() - t0_total

results_dt = pd.DataFrame({
    "fold": np.arange(1, len(dt_fold_times) + 1),
    "train_time_s": dt_fold_times,
    "roc_auc": dt_fold_auc,
    "accuracy": dt_fold_acc,
    "f1": dt_fold_f1
})

print("Decision Tree — per-fold results:")
display(results_dt)

print("\nDecision Tree — averages:")
print(f"Avg ROC AUC:   {np.mean(dt_fold_auc):.4f}")
print(f"Avg Accuracy:  {np.mean(dt_fold_acc):.4f}")
print(f"Avg F1:        {np.mean(dt_fold_f1):.4f}")

print("\nDecision Tree — training time:")
print(f"Total training time (all folds): {total_train_time:.3f}s")
print(f"Avg training time per fold:      {np.mean(dt_fold_times):.3f}s")